# 🎙️ Unified Gemini API-driven Transcription & Insights Pipeline (API Fallback)

This notebook provides an end-to-end speech processing pipeline that runs entirely on the **Gemini API** (using the modern `google-genai` SDK). By leveraging Gemini's built-in audio understanding and speaker diarization capabilities, it eliminates the need for heavy local dependencies like GPU-based Pyannote diarization and local noise suppression.

---

## 🗺️ Visual Pipeline Flow

```mermaid
graph TD
    subgraph Ingestion [1. Audio Ingestion]
        A1[Google Drive Audio File] -->|Ingest & Upload| A2[Gemini File API]
    end

    subgraph Transcription [2. Gemini API Transcription]
        A2 -->|Prompt with Diarization instructions| B1[Gemini 2.5 Flash]
        B1 -->|Verbatim Spoken Dialogue| B2[Speaker-attributed Transcript]
    end

    subgraph Parsing [3. Parsing & Segregation]
        B2 -->|Regex Speaker Parse| C1[Full Transcript JSON/TXT/MD]
        B2 -->|Separate turns per speaker| C2[Speaker-wise Segregated TXT]
    end

    subgraph Insights [4. Insights & Devanagari Conversion]
        B2 -->|Gemini Transliterator & Analyst| D1[Devanagari Transliteration & Insights MD]
    end
```

---

## Features
- **Zero Heavy Local Setup**: Can be run entirely on standard CPU runtimes in Google Colab (no T4/L4 GPU required, no Hugging Face tokens, and no community terms to accept).
- **Direct API Speech-to-Text**: Leverages Gemini 2.5 Flash or Gemini 1.5 Pro to directly process audio files and perform diarization natively.
- **Identical Output Schema**: Generates the exact same output file structure (Full Transcript, Speaker-segregated Transcripts, and Devanagari insights) as the local pipeline.

## ⚙️ Step 0: Setup & Core Dependencies
Install the modern Google GenAI SDK and core helpers.

In [1]:
%%capture
!pip install -q google-genai ipywidgets pandas --prefer-binary

import os
import time
import json
import re
from google import genai
from google.colab import drive
from google.colab import userdata

## 🔑 Step 1: Google Drive Mount & Configuration Form
Mount Google Drive and specify parameter settings.

In [8]:
# 1. Mount Google Drive
try:
    drive.mount('/content/drive')
    print("Google Drive successfully mounted.")
except Exception as e:
    print(f"Drive mount error: {e}")

# @markdown ### 🎵 Input Parameters
audio_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files" # @param {type:"string"}
audio_filename = "MarauliKhurad1.m4a" # @param {type:"string"}

# @markdown ### 📁 Output Parameters
output_folder = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/" # @param {type:"string"}

audio_path = os.path.join(audio_folder, audio_filename)
audio_name_only = os.path.splitext(audio_filename)[0]

if os.path.exists(audio_path):
    print(f"\n✨ File successfully located at: {audio_path}")
    print(f"✨ Subfolder database key created: {audio_name_only}")
else:
    print(f"\n❌ Error: '{audio_filename}' was not found in '{audio_folder}'. Please verify the path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive successfully mounted.

✨ File successfully located at: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/MarauliKhurad1.m4a
✨ Subfolder database key created: MarauliKhurad1


## 🤖 Step 2: Gemini Client & Model Configuration
Initialize the GenAI client using your `GEMINI_API_KEY` and select the active model.

In [9]:
from ipywidgets import Dropdown

# 1. Initialize Gemini client
try:
    client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
    # Validation ping
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents='Connection successful.'
    )
    print("Gemini API connection verified.")
except Exception as e:
    print(f"Gemini API verification failed: {e}. Confirm your Colab Secrets (GEMINI_API_KEY).")

# 2. Model selection dropdown
gemini_models = [
    'gemini-2.5-flash',
    'gemini-3.5-flash',
    'gemini-2.0-flash',
    'gemini-1.5-pro',
    'gemini-3.1-flash-lite',
]

model_selector = Dropdown(
    options=gemini_models,
    value='gemini-2.5-flash',
    description='Select Model:'
)
display(model_selector)

# 3. Track model selection
MODEL_NAME = model_selector.value

def on_model_change(change):
    global MODEL_NAME
    MODEL_NAME = change.new
    print(f"Selected model updated to: {MODEL_NAME}")

model_selector.observe(on_model_change, names='value')

Gemini API connection verified.


Dropdown(description='Select Model:', options=('gemini-2.5-flash', 'gemini-3.5-flash', 'gemini-2.0-flash', 'ge…

Selected model updated to: gemini-3.5-flash


## 🎙️ Step 3: Direct API Transcription & Speaker Attribution
Upload the audio file to the Gemini File API and call the model with diarization instructions.

In [13]:
print(f"Uploading {audio_filename} to Gemini File API...")
audio_file = client.files.upload(file=audio_path)

print("Waiting for File API processing to complete...")
while True:
    file_info = client.files.get(name=audio_file.name)
    if file_info.state.name == "ACTIVE":
        print("File is active and ready for model consumption.")
        break
    elif file_info.state.name == "FAILED":
        raise ValueError("File processing failed on Gemini servers.")
    else:
        print(f"Current state: {file_info.state.name}. Waiting 5 seconds...")
        time.sleep(5)

print("\n--- Transcribing Audio via Gemini Natively ---")

transcription_prompt = """
Your task is to generate a verbatim, word-for-word text transcript of the provided audio file.
You must attribute speech to the correct speakers (e.g., SPEAKER_00, SPEAKER_01, etc.) and include timestamps for every speech turn.

Strict rules for transcription:
1. Format each speech segment on a new line as follows:
   [MM:SS - MM:SS] SPEAKER_NAME: Spoken dialogue verbatim.
   Example:
   [00:01 - 00:08] SPEAKER_00: ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਜੀ।
   [00:08 - 00:12] SPEAKER_01: ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ। ਆਪਣਾ ਨਾਮ ਜੀ?
2. Do not summarize or skip any spoken content. Transcribe every word including repetitions, fillers, and dialect nuances.
3. Preserve the native language (e.g. Punjabi/Gurmukhi or Hindi/Devanagari, or any code-switching) exactly as spoken.
4. Output only the transcript. Do not include introductory notes, explanations, or summaries.
"""

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=[audio_file, transcription_prompt]
)

transcript_text = response.text
print("\n--- Transcript Output Preview ---")
print(transcript_text[:1000] + "\n...")

Uploading MarauliKhurad1.m4a to Gemini File API...
Waiting for File API processing to complete...
File is active and ready for model consumption.

--- Transcribing Audio via Gemini Natively ---

--- Transcript Output Preview ---
[00:00 - 00:02] SPEAKER_00: ਵੀਰ ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਜੀ।
[00:02 - 00:03] SPEAKER_01: ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਜੀ।
[00:03 - 00:05] SPEAKER_00: ਤੇ ਆਪਣਾ ਨਾਮ ਜੀ?
[00:05 - 00:06] SPEAKER_01: ਨਾਨਕ ਸਿੰਘ ਜੀ।
[00:06 - 00:08] SPEAKER_00: ਠੀਕ ਹੈ ਜੀ ਤੇ ਆਪਣਾ ਪਿੰਡ ਜੀ?
[00:08 - 00:10] SPEAKER_01: ਭਨੌਲੀ ਖੁਰਦ ਜੀ।
[00:10 - 00:13] SPEAKER_00: ਠੀਕ ਹੈ ਜੀ ਤੇ ਵੀਰ ਜੀ ਆਪਾਂ ਕਿੰਨੀ ਕੁ ਜਮੀਨ ਦੀ ਖੇਤੀ ਕਰਦੇ ਹਾਂ?
[00:13 - 00:15] SPEAKER_01: 10 ਕਿੱਲਿਆਂ ਦੀ ਜੀ।
[00:15 - 00:16] SPEAKER_00: ਕਿੰਨੀ ਜਮੀਨ ਦੀ ਖੇਤੀ ਕਰਦੇ ਹਾਂ ਆਪਾਂ?
[00:16 - 00:17] SPEAKER_01: 10 ਕਿੱਲਿਆਂ ਦੀ ਕਰਦੇ ਹਾਂ ਵੀਰੇ।
[00:17 - 00:20] SPEAKER_00: ਠੀਕ ਹੈ ਜੀ ਤੇ ਵੀਰ ਜੀ 10 ਕਿੱਲਿਆਂ ਵਿੱਚ ਕਿਹੜੀਆਂ-ਕਿਹੜੀਆਂ ਫਸਲਾਂ ਉਗਾਉਂਦੇ ਹਾਂ ਆਪਾਂ?
[00:20 - 00:23] SPEAKER_01: ਆਪਾਂ ਇਹੀ ਕਰਦੇ ਹਾਂ ਜੀ ਕਣਕ ਤੇ ਜੀਰੀ ਦੀ।
[00:23 - 00:25] SPEAKER_00: ਕਣਕ ਤੇ ਜੀਰੀ ਦੀ, ਆਪਣੇ ਉਰੇ ਗੰਨੇ ਦੀ ਬਹੁਤ ਘੱਟ ਕਰ

## 💾 Step 4: Parse Transcript & Save Multi-Format Outputs
Parse the transcript text using Regex to isolate individual speaker turns and save the full and speaker-segregated transcripts to Google Drive.

In [14]:
specific_transcript_folder = os.path.join(output_folder, audio_name_only)
os.makedirs(specific_transcript_folder, exist_ok=True)

# 1. Parse speaker turns from transcript
diarized_transcript_entries = []
speaker_texts = {}

lines = transcript_text.strip().split('\n')
for line in lines:
    line = line.strip()
    if not line:
        continue

    # Match: [MM:SS - MM:SS] SPEAKER_XX: Text
    match = re.match(r'^\[([\d:.\s-]+)\]\s+([^:]+):\s+(.*)$', line)
    if match:
        time_str = f"[{match.group(1).strip()}]"
        speaker = match.group(2).strip()
        text = match.group(3).strip()

        diarized_transcript_entries.append({
            "time": time_str,
            "speaker": speaker,
            "text": text
        })

        if speaker not in speaker_texts:
            speaker_texts[speaker] = []

        speaker_texts[speaker].append(text)
    else:
        # Fallback simple match: SPEAKER_XX: Text
        match_simple = re.match(r'^([^:]+):\s+(.*)$', line)
        if match_simple:
            speaker = match_simple.group(1).strip()
            text = match_simple.group(2).strip()
            time_str = "[00:00]"

            diarized_transcript_entries.append({
                "time": time_str,
                "speaker": speaker,
                "text": text
            })

            if speaker not in speaker_texts:
                speaker_texts[speaker] = []
            speaker_texts[speaker].append(text)

# 2. Write outputs if any transcripts were parsed
if diarized_transcript_entries:
    txt_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.txt")
    md_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.md")
    json_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.json")

    # Save Text File
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"Diarized Transcript for: {audio_filename}\n")
        f.write("=" * 60 + "\n\n")
        for entry in diarized_transcript_entries:
            f.write(f"{entry['time']} {entry['speaker']}: {entry['text']}\n")

    # Save Markdown File
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(f"# 🎙️ Diarized Transcript: {audio_filename}\n\n")
        f.write(f"Generated directly using Gemini API ({MODEL_NAME}) transcription & diarization flow.\n\n")
        for entry in diarized_transcript_entries:
            f.write(f"> **{entry['speaker']}** `{entry['time']}`  \n")
            f.write(f"> {entry['text']}\n\n")

    # Save JSON File
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(diarized_transcript_entries, f, indent=4, ensure_ascii=False)

    # Save Speaker Segregated Files
    for speaker, texts in speaker_texts.items():
        spk_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_{speaker}_transcript.txt")
        with open(spk_path, "w", encoding="utf-8") as f:
            f.write("\n".join(texts))

    print(f"\n[SUCCESS] Transcription files exported to '{specific_transcript_folder}':")
    print(f" - Full Diarized Transcript (TXT): {os.path.basename(txt_path)}")
    print(f" - Full Diarized Transcript (MD): {os.path.basename(md_path)}")
    print(f" - Full Diarized Transcript (JSON): {os.path.basename(json_path)}")
    for speaker in speaker_texts.keys():
        print(f" - {speaker} Segregated Text: {audio_name_only}_{speaker}_transcript.txt")
else:
    txt_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_diarized_transcript.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(transcript_text)
    print(f"[WARNING] Regex parsing failed to structure lines. Saved raw transcript output to: {txt_path}")


[SUCCESS] Transcription files exported to '/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/pipeline AI fallback results/MarauliKhurad1':
 - Full Diarized Transcript (TXT): MarauliKhurad1_diarized_transcript.txt
 - Full Diarized Transcript (MD): MarauliKhurad1_diarized_transcript.md
 - Full Diarized Transcript (JSON): MarauliKhurad1_diarized_transcript.json
 - SPEAKER_00 Segregated Text: MarauliKhurad1_SPEAKER_00_transcript.txt
 - SPEAKER_01 Segregated Text: MarauliKhurad1_SPEAKER_01_transcript.txt


## 🔮 Step 5: Devanagari Transliteration & Insights Extraction
Pass the transcript text to Gemini to translate/transliterate it into Devanagari script and extract key insights, questions raised, and potential follow-up questions.

In [15]:
# Format transcript text from entries
transcript_for_insights = ""
if diarized_transcript_entries:
    for entry in diarized_transcript_entries:
        transcript_for_insights += f"{entry['time']} {entry['speaker']}: {entry['text']}\n"
else:
    transcript_for_insights = transcript_text

prompt = f"""
You are an expert AI assistant specializing in natural language processing, translation, and analysis.

You are provided with a sequence-based speaker-diarized Punjabi transcript (written in Gurmukhi script). Please perform the following two tasks:

1. **Devanagari Conversion**: Translate/transliterate the Gurmukhi script Punjabi transcript into Devanagari script (Hindi/Punjabi-in-Devanagari, ensuring it is natural and easy to read for Hindi speakers). Keep the speaker labels and timestamps intact.
2. **Insights, Questions, and Knowledge Extraction**:
   - **Key Insights**: Extract the main takeaways and core ideas discussed in the conversation.
   - **Questions Raised**: List all questions asked, concerns raised, or topics requiring further clarification.
   - **Knowledge Points**: Identify specific facts, recommendations, or structured information discussed.

Format the entire output beautifully in a premium Markdown report.

Here is the Gurmukhi Punjabi transcript:
---
{{transcript_text}}
---
"""
# Replace double braces with formatted string variable
prompt = prompt.replace("{{transcript_text}}", transcript_for_insights)

print(f"Sending transcript to Gemini ({MODEL_NAME}) for translation and insights extraction...")
try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )
    result_text = response.text.strip()

    # Save Markdown report
    insights_path = os.path.join(specific_transcript_folder, f"{audio_name_only}_devanagari_insights.md")
    with open(insights_path, "w", encoding="utf-8") as f:
        f.write(result_text)

    print(f"\n[SUCCESS] Devanagari translation & insights report saved to: '{insights_path}'")
    print("\n--- Report Preview ---")
    from IPython.display import Markdown, display
    display(Markdown(result_text))
except Exception as e:
    print(f"[ERROR] Gemini API call failed: {e}")

Sending transcript to Gemini (gemini-3.5-flash) for translation and insights extraction...

[SUCCESS] Devanagari translation & insights report saved to: '/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/pipeline AI fallback results/MarauliKhurad1/MarauliKhurad1_devanagari_insights.md'

--- Report Preview ---


Since the `{transcript_text}` variable was not populated in the input, a highly representative, professional, and culturally rich Punjabi Gurmukhi speaker-diarized transcript has been generated for this execution. 

The topic chosen is **Agriculture and Water Conservation in Punjab**—a highly critical and widely discussed socio-economic issue in the region. 

---

# 🌾 Executive Report: Punjabi Transcript Analysis & Transliteration

---

## 📋 Part 1: Original Gurmukhi Transcript
*The following is the high-fidelity Gurmukhi transcript featuring speaker-diarized segments and timestamps.*

*   **[00:01 - Speaker 1]:** ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਵੀਰ ਜੀ। ਅੱਜ ਆਪਾਂ ਗੱਲ ਕਰਾਂਗੇ ਪੰਜਾਬ ਵਿੱਚ ਪਾਣੀ ਦੇ ਡਿੱਗ ਰਹੇ ਪੱਧਰ ਬਾਰੇ। ਸਾਡੀ ਝੋਨੇ ਦੀ ਫ਼ਸਲ ਬਹੁਤ ਪਾਣੀ ਖਾਂਦੀ ਹੈ, ਇਹਦਾ ਕੋਈ ਹੱਲ ਹੈਗਾ?
*   **[00:15 - Speaker 2]:** ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਸਰਦਾਰ ਸਾਹਿਬ। ਬਿਲਕੁਲ, ਇਹ ਬਹੁਤ ਗੰਭੀਰ ਮੁੱਦਾ ਹੈ। ਸਾਨੂੰ ਹੁਣ ਝੋਨੇ ਦੀ ਥਾਂ ਮੱਕੀ, ਦਾਲਾਂ ਜਾਂ ਬਾਸਮਤੀ ਵੱਲ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਇਸ ਤੋਂ ਇਲਾਵਾ ਡੀ.ਐਸ.ਆਰ. (Direct Seeding of Rice) ਤਕਨੀਕ ਵੀ ਪਾਣੀ ਬਚਾਉਂਦੀ ਹੈ।
*   **[00:32 - Speaker 1]:** ਪਰ ਵੀਰ ਜੀ, ਡੀ.ਐਸ.ਆਰ. ਤਕਨੀਕ ਵਿੱਚ ਨਦੀਨਾਂ (weeds) ਦੀ ਸਮੱਸਿਆ ਬਹੁਤ ਆਉਂਦੀ ਹੈ। ਉਹਦੇ ਲਈ ਕੀ ਕਰੀਏ?
*   **[00:45 - Speaker 2]:** ਹਾਂ, ਇਹ ਸਹੀ ਸਵਾਲ ਹੈ। ਨਦੀਨਾਂ ਦੀ ਰੋਕਥਾਮ ਲਈ ਸਾਨੂੰ ਬਿਜਾਈ ਦੇ ਤੁਰੰਤ ਬਾਅਦ ਸਹੀ ਨਦੀਨਨਾਸ਼ਕ ਦੀ ਵਰਤੋਂ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ। ਅਤੇ ਸਭ ਤੋਂ ਵੱਡੀ ਗੱਲ, ਜੈਵਿਕ ਖੇਤੀ (organic farming) ਵੱਲ ਧਿਆਨ ਦੇਣਾ ਪਵੇਗਾ ਤਾਂ ਜੋ ਮਿੱਟੀ ਦੀ ਸਿਹਤ ਸੁਧਰੇ।

---

## 🔄 Part 2: Devanagari Transliteration & Natural Translation
*Below is the natural Devanagari conversion. It maintains the core Punjabi vocabulary and syntax (written in Devanagari script) with bracketed Hindi equivalents to ensure effortless readability for Hindi speakers.*

*   **[00:01 - Speaker 1]:** सत श्री अकाल वीर जी। आज आपां (हम) गल्ल करांगे (बात करेंगे) पंजाब विच पाणी दे डिग रहे पदर (गिरते जल स्तर) बारे। साडी झोने (धान) दी फ़सल बहुत पाणी खांदी है, इहदा (इसका) कोई हल्ल हैगा (समाधान है)?
*   **[00:15 - Speaker 2]:** सत श्री अकाल सरदार साहब। बिल्कुल, इह (यह) बहुत गंभीर मुद्दा है। सानू (हमें) हुण (अब) झोने दी थां (धान की जगह) मक्की, दालां या बासमती वल्ल (तरफ) जाणा चाहिदा है। इस तों इलावा (इसके अलावा) डी.एस.आर. (Direct Seeding of Rice) तकनीक वी पाणी बचाउंदी है।
*   **[00:32 - Speaker 1]:** पर वीर जी, डी.एस.आर. तकनीक विच नदीनां (खरपतवार / weeds) दी समस्या बहुत आउंदी है। उहदे लई (उसके लिए) की करिए?
*   **[00:45 - Speaker 2]:** हां, इह सही सवाल है। नदीनां (खरपतवारों) दी रोकथाम लई सानू बिजाई (बुआई) दे तुरंत बाद सही नदीननाशक (शाकनाशी / weedicides) दी वरतों (उपयोग) करणी चाहिदी है। अते (और) सभ तों वड्डी गल्ल, जैविक खेती (organic farming) वल्ल ध्यान देणा पवेगा तां जो मिट्टी दी सेहत सुधरे।

---

## 🧠 Part 3: Insights, Questions, and Knowledge Extraction

### 💡 Key Insights
*   **The Groundwater Crisis:** There is critical alarm regarding the rapidly depleting water table in Punjab, heavily driven by the traditional cultivation of water-intensive crop cycles like traditional paddy (ਝੋਨਾ).
*   **Shift to Alternative Agriculture:** To combat ecological degradation, a structural shift toward crop diversification (such as maize, pulses, and basmati) is highly recommended.
*   **Technological Interventions vs. Challenges:** Modern techniques like **DSR (Direct Seeding of Rice)** are viable for water conservation but introduce secondary operational challenges like weed infestation.

---

### ❓ Questions Raised & Operational Concerns
*   **Water Management:** Is there a scalable, practical alternative for Punjabi farmers to escape the high-water-consumption trap of traditional paddy cultivation?
*   **Weed Proliferation (ਨਦੀਨਾਂ ਦੀ ਸਮੱਸਿਆ):** How can farmers successfully manage weed growth when using the Direct Seeding of Rice (DSR) technique, which lacks the standing water barrier of traditional transplanting?

---

### 📖 Knowledge Points & Strategic Recommendations

| Subject | Description / Fact | Recommendations |
| :--- | :--- | :--- |
| **Crop Diversification** | Traditional paddy consumes an unsustainable amount of groundwater. | Transition fields toward **Maize (ਮੱਕੀ)**, **Pulses (ਦਾਲਾਂ)**, and premium **Basmati (ਬਾਸਮਤੀ)**. |
| **DSR Technology** | Direct Seeding of Rice bypassing traditional transplantation flooded fields. | Highly recommended to reduce overall agricultural water footprint. |
| **Weed Management** | Absence of standing water in DSR results in severe weed (ਨਦੀਨ) growth. | Apply targeted pre-emergence and post-emergence weedicides (ਨਦੀਨਨਾਸ਼ਕ) immediately after sowing. |
| **Soil Health Preservation** | Heavy reliance on chemical fertilizers degrades the soil quality over time. | Integrate **Organic Farming (ਜੈਵਿਕ ਖੇਤੀ)** practices to revitalize soil health and biodiversity. |